In [9]:
# Module 1 — Đo VRAM trước khi load model (baseline)
!nvidia-smi

# Ước lượng nhanh: weights ≈ số_param (tỷ) × bytes/param
# FP16/BF16 = 2 byte/param; INT8 = 1; INT4 ≈ 0.5
# Cộng thêm KV-cache + activation + overhead vLLM (thường +30–80% tùy max_model_len)
PARAMS_B = 3.0  # Qwen2.5-3B ≈ 3B params — sửa nếu đổi model
bytes_fp16 = PARAMS_B * 2  # GB cho weights FP16
print(f"\nƯớc lượng weights FP16 ~ {bytes_fp16:.1f} GB")
print("T4 16GB: 3B FP16 thường ổn; 7B FP16 sát trần → cân nhắc AWQ/INT4 (M1).")
print("Chạy lại nvidia-smi SAU health-check để so VRAM thực tế vs ước lượng.")

Mon Nov  2 15:26:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   67C    P0             30W /   70W |   13439MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Deploy Qwen bằng vLLM và Ngrok (Module 1 Serving + Module 3 stream)

Notebook này giúp bạn:
1. Đo VRAM trước/sau load — so với ước lượng kích thước model.
2. Serve bằng **vLLM FP16** + tham số serve (`dtype`, `max_model_len`, `gpu-memory-utilization`).
3. Public API bằng Ngrok (OpenAI-compatible).
4. Gọi chat / **stream** với parameters client (`temperature`, `top_p`, `max_tokens`).

In [10]:
# Cài đặt các thư viện cần thiết
!pip install vllm pyngrok

### Cấu hình Ngrok Authtoken
Bạn cần truy cập [Ngrok Dashboard](https://dashboard.ngrok.com/), đăng ký tài khoản miễn phí và lấy **Authtoken** điền vào bên dưới.

In [ ]:
from google.colab import userdata
from pyngrok import ngrok
from pyngrok.exception import PyngrokNgrokError
import os

# Cách 1: Sử dụng Secrets của Colab
try:
    NGROK_TOKEN = userdata.get("NGROK_AUTHTOKEN")
except Exception:
    # Cách 2: Điền trực tiếp Token của bạn
    NGROK_TOKEN = "2ok7kIrwCusDbdNqEbmgxnvuwyF_5XvMNnwCDGtVrQAWyrhfc"

if not NGROK_TOKEN:
    print("CẢNH BÁO: Vui lòng cấu hình NGROK_TOKEN!")
else:
    try:
        ngrok.set_auth_token(NGROK_TOKEN)
        print("Đã cấu hình Ngrok Authtoken thành công.")
    except PyngrokNgrokError as e:
        if "already set" in str(e):
            print("Ngrok Authtoken đã được cấu hình từ trước. Sẵn sàng sử dụng!")
        else:
            raise e

Ngrok Authtoken đã được cấu hình từ trước. Sẵn sàng sử dụng!


### Khởi chạy vLLM Server (FP16 + parameters serve)

Chạy background cổng `8000`. Theo Module 1:
- **FP16** (`--dtype float16`): cân bằng chất lượng/VRAM trên GPU NVIDIA (T4).
- **`max_model_len`**: giới hạn context → giảm KV-cache (VRAM).
- **`gpu-memory-utilization`**: phần VRAM vLLM được dùng (KV + weights).
- Model lớn hơn / VRAM thiếu → đổi sang AWQ/INT4 (không dùng ô này).

In [12]:
import subprocess

# Checkpoint — phải khớp tên khi gọi /v1/chat/completions
model_name = "Qwen/Qwen2.5-3B-Instruct"

# --- Quantization / dtype (Module 1) ---
# float16 = FP16 serving trên GPU NVIDIA. Đổi "auto" nếu muốn vLLM tự chọn.
DTYPE = "float16"

# --- Parameters khi serve ---
PORT = 8000
MAX_MODEL_LEN = 4096          # context window; giảm nếu OOM
GPU_MEM_UTIL = 0.90           # 0.85–0.95 tùy model
MAX_NUM_SEQS = 8              # số request đồng thời (Colab T4: giữ thấp)

cmd = (
    "python3 -m vllm.entrypoints.openai.api_server "
    f"--model {model_name} "
    f"--port {PORT} "
    f"--dtype {DTYPE} "
    f"--max-model-len {MAX_MODEL_LEN} "
    f"--gpu-memory-utilization {GPU_MEM_UTIL} "
    f"--max-num-seqs {MAX_NUM_SEQS}"
)

with open("vllm.log", "w") as log_file:
    process = subprocess.Popen(cmd, shell=True, stdout=log_file, stderr=log_file)

print("Đang khởi động vLLM (FP16) ở background...")
print(f"  model={model_name}")
print(f"  dtype={DTYPE}  max_model_len={MAX_MODEL_LEN}  gpu_mem={GPU_MEM_UTIL}  max_num_seqs={MAX_NUM_SEQS}")
print("Đợi 1–3 phút rồi chạy health-check + nvidia-smi sau load.")

Đang khởi động vLLM Server ở background... Vui lòng đợi khoảng 1-2 phút để tải model.


### Kích hoạt Ngrok Tunnel
Đoạn code này sẽ tạo một public URL để bạn truy cập từ xa.

In [13]:
# Mở tunnel kết nối tới port 8000
public_url = ngrok.connect(8000)
print("\n--- ĐÃ HOÀN THÀNH DEPLOY ---")
print(f"Địa chỉ public API của bạn: {public_url}")
print("Bạn có thể dùng URL này làm 'base_url' cho OpenAI client SDK (ví dụ: {public_url}/v1)")


--- ĐÃ HOÀN THÀNH DEPLOY ---
Địa chỉ public API của bạn: NgrokTunnel: "https://950b-34-12-92-131.ngrok-free.app" -> "http://localhost:8000"
Bạn có thể dùng URL này làm 'base_url' cho OpenAI client SDK (ví dụ: {public_url}/v1)


### Theo dõi log của vLLM
Chạy cell bên dưới để kiểm tra tiến trình tải model và nhận request.

In [14]:
# Xem 20 dòng log mới nhất
!tail -n 20 vllm.log

### Health check + đo VRAM sau khi load
Chờ `/health` OK rồi so VRAM thực tế với ước lượng FP16 ở cell đầu (weights ≈ params×2 GB + KV-cache).

In [15]:
import subprocess
import time

import requests

local_health_url = "http://localhost:8000/health"
print("Đang chờ vLLM sẵn sàng (có thể 2–4 phút)...\n")

ready = False
for i in range(60):
    try:
        res = requests.get(local_health_url, timeout=2)
        if res.status_code == 200:
            print("\n[SẴN SÀNG] vLLM đã lên /health=200")
            ready = True
            break
    except requests.exceptions.ConnectionError:
        pass

    with open("vllm.log", "r") as f:
        lines = f.readlines()
        last_line = lines[-1].strip() if lines else "Chưa có log..."
        print(f"[Đợi {i * 5}s] log: {last_line[:100]}...")
    time.sleep(5)

if not ready:
    print("\n[CẢNH BÁO] Timeout 5 phút — xem `!tail -n 50 vllm.log`.")
else:
    # So VRAM sau load với ước lượng FP16 (Module 1)
    print("\n=== VRAM SAU KHI LOAD (so với ước lượng cell đầu) ===")
    smi = subprocess.check_output(
        [
            "nvidia-smi",
            "--query-gpu=name,memory.used,memory.total,utilization.gpu",
            "--format=csv,noheader,nounits",
        ],
        text=True,
    ).strip()
    print(smi)
    used_mib = float(smi.split(",")[1].strip())
    total_mib = float(smi.split(",")[2].strip())
    params_b = 3.0  # khớp PARAMS_B / model_name
    weights_fp16_gb = params_b * 2
    used_gb = used_mib / 1024
    print(f"Đã dùng ≈ {used_gb:.2f} GB / {total_mib/1024:.1f} GB")
    print(f"Ước lượng weights FP16 ≈ {weights_fp16_gb:.1f} GB")
    print(
        f"Phần dư ≈ {used_gb - weights_fp16_gb:.2f} GB "
        "(KV-cache + runtime vLLM — tăng nếu max_model_len lớn)."
    )
    if used_gb > total_mib / 1024 * 0.92:
        print("⚠️ VRAM gần đầy — giảm MAX_MODEL_LEN hoặc dùng quant INT4/AWQ.")

Đang kiểm tra tiến trình khởi động của vLLM (Quá trình này có thể mất 2-4 phút)...


[SẴN SÀNG] vLLM đã khởi động xong hoàn toàn và đang hoạt động tốt!


### Thử nghiệm (Test) API của bạn
Dưới đây là 2 cách để bạn kiểm tra xem API hoạt động chính xác chưa.

#### Cách 1: Sử dụng thư viện `requests` (Không cần cài đặt gì thêm)

In [17]:
import requests
import json

# Lấy URL ngrok động từ biến đã lưu ở trên
api_url = f"{public_url.public_url}/v1/chat/completions"

headers = {
    "Content-Type": "application/json",
    "ngrok-skip-browser-warning": "1",
}

# Parameters khi gọi (Module 1): temperature / top_p / max_tokens
data = {
    "model": model_name,
    "messages": [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Hãy viết một bài thơ 4 câu nói về lập trình máy tính."},
    ],
    "temperature": 0.7,
    "top_p": 0.9,
    "max_tokens": 256,
    "stream": False,
}

print(f"Đang gửi request tới: {api_url} ...")
try:
    response = requests.post(api_url, headers=headers, json=data)
    if response.status_code == 200:
        result = response.json()
        print("\n--- Phản hồi từ Model ---")
        print(result['choices'][0]['message']['content'])
    else:
        print(f"Lỗi: {response.status_code}")
        print(response.text)
except Exception as e:
    print(f"Không thể kết nối tới server: {e}")

Đang gửi request tới: https://950b-34-12-92-131.ngrok-free.app/v1/chat/completions ...

--- Phản hồi từ Model ---
Trên màn hình đen trắng,  
Mã lệnh bay trong không gian,  
Như con bướm tìm đường,  
Tạo nên thế giới ảo.


#### Stream chat (chuẩn OpenAI / vLLM — Module 3 Bài 4)

Thêm `"stream": true`. vLLM trả từng dòng `data: {...}` rồi `data: [DONE]`. Token hiện dần, không chờ cả câu.

In [ ]:
import json
import requests

api_url = f"{public_url.public_url}/v1/chat/completions"
headers = {
    "Content-Type": "application/json",
    "ngrok-skip-browser-warning": "1",
}
payload = {
    "model": model_name,
    "messages": [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Hãy viết một bài thơ 4 câu nói về lập trình máy tính."},
    ],
    "temperature": 0.7,
    "top_p": 0.9,
    "max_tokens": 256,
    "stream": True,
}

print(f"Streaming từ: {api_url}\n---")
with requests.post(api_url, headers=headers, json=payload, stream=True, timeout=120) as resp:
    resp.raise_for_status()
    for raw in resp.iter_lines(decode_unicode=True):
        if not raw:
            continue
        if raw.startswith("data: "):
            raw = raw[6:]
        if raw.strip() == "[DONE]":
            print("\n--- [DONE] ---")
            break
        chunk = json.loads(raw)
        delta = chunk["choices"][0].get("delta") or {}
        token = delta.get("content") or ""
        if token:
            print(token, end="", flush=True)

#### Cách 2: Sử dụng thư viện `openai` SDK

In [ ]:
# pip đã có openai (dependency của vllm). base_url phải kết thúc bằng /v1
from openai import OpenAI

client = OpenAI(
    base_url=f"{public_url.public_url}/v1",
    api_key="EMPTY",  # vLLM local không cần key thật
)

print("--- OpenAI SDK stream ---")
stream = client.chat.completions.create(
    model=model_name,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Giải thích PagedAttention trong một đoạn ngắn."},
    ],
    temperature=0.7,
    top_p=0.9,
    max_tokens=256,
    stream=True,
)
for event in stream:
    token = event.choices[0].delta.content or ""
    if token:
        print(token, end="", flush=True)
print("\n--- xong ---")

print("\nBase URL public (dùng trên máy local):")
print(public_url.public_url)